# Data Preprocessing 
This script is responsible for preprocessing the raw cryptocurrency price data. It performs the following steps:
1. Loads the raw data from a CSV file.
2. Cleans the data by handling missing values and outliers.
3. Scales the features using StandardScaler.
4. Saves the cleaned and processed data to a new CSV file for further analysis.


## Required Libraries

In [ ]:
import pandas as pd
import numpy as np 
from sklearn.preprocessing import StandardScaler
import os

## CONFIGURATION

In [ ]:
RAW_DATA_PATH = '../data/raw/crypto_prices.csv'

# Load Datasets

In [3]:
df = pd.read_csv(RAW_DATA_PATH)
print("Data loaded successfully. Shape:", df.shape)
print("info:")
print(df.info())
print("---" * 20)
print("head:")
print(df.head())
print("---" * 20)
print("describe:")
print(df.describe())


Data loaded successfully. Shape: (72946, 10)
info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72946 entries, 0 to 72945
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   72946 non-null  int64  
 1   open         72946 non-null  float64
 2   high         72946 non-null  float64
 3   low          72946 non-null  float64
 4   close        72946 non-null  float64
 5   volume       72946 non-null  float64
 6   marketCap    72946 non-null  float64
 7   timestamp    72946 non-null  object 
 8   crypto_name  72946 non-null  object 
 9   date         72946 non-null  object 
dtypes: float64(6), int64(1), object(3)
memory usage: 5.6+ MB
None
------------------------------------------------------------
head:
   Unnamed: 0        open        high         low       close  volume  \
0           0  112.900002  118.800003  107.142998  115.910004     0.0   
1           1    3.493130    3.692460    3.346060    3.5

### Here we can see a unnamed column are available so we have to remove
## Drop Unnecessary Column

In [4]:
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72946 entries, 0 to 72945
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   open         72946 non-null  float64
 1   high         72946 non-null  float64
 2   low          72946 non-null  float64
 3   close        72946 non-null  float64
 4   volume       72946 non-null  float64
 5   marketCap    72946 non-null  float64
 6   timestamp    72946 non-null  object 
 7   crypto_name  72946 non-null  object 
 8   date         72946 non-null  object 
dtypes: float64(6), object(3)
memory usage: 5.0+ MB


# Fixed Datetime Columns

In [6]:
#Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Drop timestamp column 
df.drop(columns = ["timestamp"], inplace = True)


# Sort Time-Series Correctly

In [7]:
df.sort_values(by = ["crypto_name", "date"], inplace = True)
df.reset_index(drop = True, inplace = True)

# Check and Handle Duplicates

In [8]:
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

df.drop_duplicates(inplace = True)

Number of duplicate rows: 0


# Missing Value Handling
N/A for this datasets

In [9]:
df.isnull().sum()

df = df.groupby("crypto_name").apply(lambda x: x.ffill()).reset_index(drop = True)

df.dropna(subset = ["close", "volume", "marketCap"], inplace = True)

# Create Log Return

In [13]:
df["log_return"] = df.groupby("crypto_name")["close"].transform(
    lambda x: np.log(x / x.shift(1))
)


# Scale numerical features

## StandardScaler
    StandardScaler is used to standardize features by removing the mean and scaling to unit variance.

It transforms data using:
𝑧 = (𝑥 − 𝜇) / 𝜎

Where:

𝑥 = original value
𝜇 = mean of the feature
𝜎 = standard deviation

After scaling:
Mean = 0
Standard deviation = 1

In [14]:
num_features = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "marketCap",
    "log_return"
]

scaler = StandardScaler()
df[num_features] = scaler.fit_transform(df[num_features])

# Save Cleaned Dataset

In [17]:
OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(OUTPUT_DIR, "cleaned_data.csv")
df.to_csv(OUTPUT_PATH, index=False)

print("Preprocessing complete!")
print("Final Shape:", df.shape)


Preprocessing complete!
Final Shape: (72946, 9)


The raw cryptocurrency dataset was cleaned by 
removing redundant columns,
converting date fields to datetime format,
handling duplicates,
and ensuring proper time-series ordering.
Log return were computed for each cryptocurrency to support volatility estimation.
numerical features were standardized to improve model convergence and performance. 